In [2]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from tqdm.auto import tqdm
from google.oauth2 import service_account
import numpy as np

# 1. 신분증(JSON 키) 경로 지정
KEY_PATH = '../google_key.json'

# 2. 인증 객체 생성
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)

# 3. 프로젝트 ID 설정
project_id = 'gdelt-analysis-494301'

# 4. 데이터 불러오기
query = "SELECT SQLDATE FROM `gdelt-bq.full.events` LIMIT 5"
df = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("인증 성공! 데이터를 가져왔습니다.")

Downloading: 100%|██████████|
인증 성공! 데이터를 가져왔습니다.


In [3]:
# 1. 고위험군 CAMEO 코드 리스트
target_cameo_codes = [
    '150', '151', '152', '153', '154', '155', 
    '190', '191', '192', '193', '194', '195', '196', 
    '200', '201', '202', '203', '204'
]
formatted_codes = ", ".join([f"'{code}'" for code in target_cameo_codes])

query = f"""
SELECT 
    SQLDATE, 
    EventCode,
    GoldsteinScale, 
    NumMentions, 
    AvgTone,
    ActionGeo_Type,
    ActionGeo_Lat, 
    ActionGeo_Long, 
    SOURCEURL
FROM `gdelt-bq.full.events`
WHERE SQLDATE >= 20130401 
  AND (
    (Actor1CountryCode = 'CHN' AND Actor2CountryCode = 'TWN') OR 
    (Actor1CountryCode = 'TWN' AND Actor2CountryCode = 'CHN')
  )
  AND EventCode IN ({formatted_codes})
  AND IsRootEvent = 1                 -- 핵심 사건만 필터링
  AND ActionGeo_Type IN (3, 4, 5)
"""

# 2. 데이터 저장
df = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("데이터 불러오기 완료")
display(df.head())


Downloading: 100%|██████████|
데이터 불러오기 완료


,SQLDATE,EventCode,GoldsteinScale,NumMentions,AvgTone,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,20260510,194,-10.0,16,-2.429197,4,24.9139,118.586,https://www.taipeitimes.com/News/front/archive...
1,20260510,154,-7.2,1,-3.183521,4,25.0478,121.532,https://news.ltn.com.tw/news/focus/breakingnew...
2,20260510,154,-7.2,1,-3.183521,4,25.0478,121.532,https://news.ltn.com.tw/news/focus/breakingnew...
3,20220320,195,-10.0,12,-1.565588,4,39.9289,116.388,https://news.webindia123.com/news/Articles/Wor...
4,20220320,195,-10.0,6,-1.565588,4,39.9289,116.388,https://news.webindia123.com/news/Articles/Wor...


In [4]:
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'].astype(str), format='%Y%m%d')

df.to_csv("01_data/raw/gdelt_raw.csv", index=False)
print(f"저장 완료: {len(df)}행")

저장 완료: 11937행


In [5]:
import pandas as pd

df = pd.read_csv("01_data/processed/final_priority.csv")

print(df[['ActionGeo_Lat', 'ActionGeo_Long']].describe())
print(f"\n좌표 결측치: {df[['ActionGeo_Lat', 'ActionGeo_Long']].isnull().sum().to_dict()}")
print(f"\n좌표 샘플:\n{df[['ActionGeo_Lat', 'ActionGeo_Long', 'priority_score']].head(10)}")

       ActionGeo_Lat  ActionGeo_Long
count     177.000000      177.000000
mean       31.823555      115.108647
std         9.472259       23.631831
min       -35.283300      -77.036400
25%        25.047800      116.388000
50%        32.526100      116.388000
75%        39.928900      121.532000
max        55.752200      149.217000

좌표 결측치: {'ActionGeo_Lat': 0, 'ActionGeo_Long': 0}

좌표 샘플:
   ActionGeo_Lat  ActionGeo_Long  priority_score
0        24.0000         119.000        0.779269
1        39.9289         116.388        0.717561
2        24.0000         119.000        0.690566
3        25.0478         121.532        0.603822
4        24.0000         119.000        0.603720
5        24.4367         118.318        0.589832
6        28.7925         117.262        0.586239
7        39.9289         116.388        0.573689
8        22.1094         120.874        0.527688
9        39.9289         116.388        0.526684


In [6]:
import pandas as pd
df = pd.read_csv("01_data/processed/spike_events.csv")
print(df.columns.tolist())
print(df.head(3))

['SQLDATE', 'DailyMentions', 'EventCount', 'AvgGoldstein', 'AvgTone', 'MA_7', 'MA_14', 'MA_30', 'MoM_rate', 'is_spike']
      SQLDATE  DailyMentions  EventCount  AvgGoldstein   AvgTone        MA_7  \
0  2018-04-19            457           5         -7.76 -2.213559  214.428571   
1  2019-01-15            868           3        -10.00 -1.978939  224.000000   
2  2019-01-16           1354           3        -10.00 -2.558135  400.285714   

        MA_14  MA_30     MoM_rate  is_spike  
0  344.642857  452.1   315.454545      True  
1  218.500000  299.9   623.333333      True  
2  283.785714  314.0  1028.333333      True  


In [8]:
import pandas as pd
df = pd.read_csv("01_data/processed/final_priority_geo.csv")
print(df.columns.tolist())

['SQLDATE', 'EventCode', 'GoldsteinScale', 'NumMentions', 'AvgTone', 'ActionGeo_Type', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL', 'score_mentions', 'score_goldstein', 'score_tone', 'score_geo', 'priority_score', 'geo_level']


In [14]:
import yaml

with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print(config['priority_score']['geo_distance_bins'])

[{'max_km': 22.2, 'score': 1.0}, {'max_km': 88.9, 'score': 0.75}, {'max_km': 177.8, 'score': 0.5}, {'max_km': 355.6, 'score': 0.25}, {'max_km': 99999, 'score': 0.1}]


In [16]:
import pandas as pd

df = pd.read_csv("01_data/processed/final_priority.csv")
print(df['score_geo'].value_counts())
print(f"\nScore 범위: {df['priority_score'].min():.3f} ~ {df['priority_score'].max():.3f}")

score_geo
0.10    299
0.50    147
1.00     30
0.25     14
0.75     13
Name: count, dtype: int64

Score 범위: 0.020 ~ 0.778


In [2]:
import pandas as pd
from datetime import datetime, timezone, timedelta

df = pd.read_csv("01_data/processed/final_priority_geo.csv")
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'])
one_year_ago = datetime.now(timezone.utc) - timedelta(days=365)
df_recent = df[df['SQLDATE'] >= one_year_ago.strftime('%Y-%m-%d')]
print(f"1년 이내 이벤트: {len(df_recent)}개")

1년 이내 이벤트: 18개


In [4]:
import pandas as pd

df = pd.read_csv("01_data/processed/satellite_passes.csv")
print(f"전체: {len(df)}건")
print(f"\n이벤트별 근접 위성 수:")
print(df.groupby('SQLDATE')['satellite_name'].count().sort_values(ascending=False))
print(f"\n최근접 거리 분포:")
print(df['min_dist_km'].describe())
print(f"\n샘플:")
print(df[['SQLDATE', 'event_lat', 'event_lon', 'satellite_name', 'min_dist_km']].head(10))

전체: 1623건

이벤트별 근접 위성 수:
SQLDATE
2025-12-30    452
2026-01-02    212
2025-12-29     96
2025-11-24     95
2026-01-01     93
2025-07-17     90
2026-04-08     87
2025-10-01     86
2025-10-29     85
2025-12-05     85
2025-07-09     84
2025-11-25     82
2025-10-08     76
Name: satellite_name, dtype: int64

최근접 거리 분포:
count    1623.000000
mean      326.628158
std       119.371269
min         9.900000
25%       242.450000
50%       348.500000
75%       426.750000
max       499.000000
Name: min_dist_km, dtype: float64

샘플:
      SQLDATE  event_lat  event_lon            satellite_name  min_dist_km
0  2025-10-01    25.0478    121.532               CARTOSAT 2A        374.3
1  2025-10-01    25.0478    121.532  CHUANG XIN 1-02(CX-1-02)        353.1
2  2025-10-01    25.0478    121.532                  DEIMOS 1        424.7
3  2025-10-01    25.0478    121.532                  ALSAT 1B        157.1
4  2025-10-01    25.0478    121.532                  GLOBAL-2        256.6
5  2025-10-01    25.0478    1

In [1]:
import pandas as pd
df = pd.read_csv("project/01_data/processed/final_priority_geo.csv")
print(df['SQLDATE'].min())
print(df['SQLDATE'].max())

2014-03-20
2026-04-08


In [4]:
import requests
from datetime import datetime
import pandas as pd

def get_cloud_cover(lat, lon, date_str):
    try:
        today = datetime.now().date()
        event_date = pd.to_datetime(date_str).date()

        if event_date < today:
            url = "https://archive-api.open-meteo.com/v1/archive"
            params = {
                "latitude": lat,
                "longitude": lon,
                "start_date": date_str,
                "end_date": date_str,
                "daily": ["cloud_cover_mean"],
                "timezone": "Asia/Tokyo"
            }
        else:
            url = "https://api.open-meteo.com/v1/forecast"
            params = {
                "latitude": lat,
                "longitude": lon,
                "current": ["cloud_cover"],
                "timezone": "Asia/Tokyo"
            }

        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if event_date < today:
            cloud = data['daily']['cloud_cover_mean'][0]
        else:
            cloud = data['current']['cloud_cover']

        return int(cloud) if cloud is not None else None
    except Exception as e:
        print(f"에러: {e}")
        return None

# 테스트
cloud = get_cloud_cover(24.0, 119.0, "2022-08-07")
print(f"구름량: {cloud}%")

구름량: 55%


In [1]:
import pandas as pd
df_spike = pd.read_csv("project/01_data/processed/spike_events.csv")
df_spike['SQLDATE'] = pd.to_datetime(df_spike['SQLDATE'])
print(df_spike['SQLDATE'].diff().value_counts().head(5))
print(df_spike.shape)
print(df_spike.head(10))

SQLDATE
1 days     8
2 days     4
3 days     3
4 days     2
26 days    1
Name: count, dtype: int64
(42, 10)
     SQLDATE  DailyMentions  EventCount  AvgGoldstein   AvgTone        MA_7  \
0 2018-04-19            457           5         -7.76 -2.213559  214.428571   
1 2019-01-15            868           3        -10.00 -1.978939  224.000000   
2 2019-01-16           1354           3        -10.00 -2.558135  400.285714   
3 2019-11-22            488           3        -10.00 -3.519337  230.000000   
4 2020-08-23            775           5        -10.00 -2.446423  183.000000   
5 2020-09-18            276           3         -7.20 -1.693657  237.428571   
6 2020-09-19            779           5         -9.44 -2.144576  318.142857   
7 2020-12-31           1384           2         -7.20 -3.243817  382.714286   
8 2021-01-19           1406           1        -10.00 -1.271789  488.142857   
9 2021-05-04            598           2        -10.00 -1.395659  168.285714   

        MA_14       MA

In [2]:
import pandas as pd
df = pd.read_csv("project/01_data/processed/final_priority_geo.csv")
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'])
print(df['SQLDATE'].diff().value_counts().head(5))
print(df.shape)
print(df.columns.tolist())

SQLDATE
0 days        32
1 days         3
1062 days      2
-1120 days     2
-657 days      2
Name: count, dtype: int64
(485, 15)
['SQLDATE', 'EventCode', 'GoldsteinScale', 'NumMentions', 'AvgTone', 'ActionGeo_Type', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL', 'score_mentions', 'score_goldstein', 'score_tone', 'score_geo', 'priority_score', 'geo_level']


In [3]:
print(df_spike.columns.tolist())
print(df_spike[['SQLDATE', 'DailyMentions', 'MoM_rate']].head(10))

['SQLDATE', 'DailyMentions', 'EventCount', 'AvgGoldstein', 'AvgTone', 'MA_7', 'MA_14', 'MA_30', 'MoM_rate', 'is_spike']
     SQLDATE  DailyMentions     MoM_rate
0 2018-04-19            457   315.454545
1 2019-01-15            868   623.333333
2 2019-01-16           1354  1028.333333
3 2019-11-22            488    16.467780
4 2020-08-23            775   919.736842
5 2020-09-18            276   245.000000
6 2020-09-19            779   264.018692
7 2020-12-31           1384   401.449275
8 2021-01-19           1406   998.437500
9 2021-05-04            598  1200.000000


In [4]:
import pandas as pd
df_raw = pd.read_csv("project/01_data/raw/gdelt_raw.csv")
df_raw['SQLDATE'] = pd.to_datetime(df_raw['SQLDATE'], format='%Y%m%d', errors='coerce')
print(df_raw.shape)
print(df_raw.columns.tolist())
print(df_raw['SQLDATE'].min(), df_raw['SQLDATE'].max())
print(df_raw['SQLDATE'].diff().value_counts().head(5))

(11937, 9)
['SQLDATE', 'EventCode', 'GoldsteinScale', 'NumMentions', 'AvgTone', 'ActionGeo_Type', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL']
NaT NaT
Series([], Name: count, dtype: int64)


In [5]:
import pandas as pd
df_raw = pd.read_csv("project/01_data/raw/gdelt_raw.csv")
print(df_raw['SQLDATE'].head(10))
print(df_raw['SQLDATE'].dtype)


0    2026-05-10
1    2026-05-10
2    2026-05-10
3    2022-03-20
4    2022-03-20
5    2021-03-20
6    2022-03-20
7    2022-03-20
8    2022-03-20
9    2022-03-20
Name: SQLDATE, dtype: object
object


In [6]:
import pandas as pd
df_raw = pd.read_csv("project/01_data/raw/gdelt_raw.csv")
df_raw['SQLDATE'] = pd.to_datetime(df_raw['SQLDATE'])

daily = df_raw.groupby('SQLDATE').agg(
    DailyMentions=('NumMentions', 'sum'),
    EventCount=('EventCode', 'count'),
    AvgGoldstein=('GoldsteinScale', 'mean'),
    AvgTone=('AvgTone', 'mean')
).reset_index().sort_values('SQLDATE')

print(daily.shape)
print(daily.head(10))
print(daily['SQLDATE'].min(), daily['SQLDATE'].max())

(2418, 5)
     SQLDATE  DailyMentions  EventCount  AvgGoldstein   AvgTone
0 2013-04-01              8           1          -9.5  3.063457
1 2013-04-14              2           2         -10.0  1.026226
2 2013-04-15              2           2         -10.0  1.107011
3 2013-04-22              2           1         -10.0  1.308901
4 2013-09-06             26           1         -10.0  1.776822
5 2013-09-08              2           1         -10.0  0.775194
6 2013-09-15              1           1         -10.0  3.985507
7 2013-09-17             10           2         -10.0  1.342282
8 2013-09-18             17           2         -10.0  1.273559
9 2013-09-20              5           1         -10.0  1.287554
2013-04-01 00:00:00 2026-05-12 00:00:00


In [3]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [2]:
df = pd.read_csv('project/01_data/raw/gdelt_raw.csv')
print('전체 행:', len(df))
print('날짜 범위:', df['SQLDATE'].min(), '~', df['SQLDATE'].max())
print('NumMentions 분포:')
print(df['NumMentions'].describe())

전체 행: 11937
날짜 범위: 2013-04-01 ~ 2026-05-12
NumMentions 분포:
count    11937.000000
mean        13.275949
std         51.319681
min          1.000000
25%          2.000000
50%          5.000000
75%         10.000000
max       1468.000000
Name: NumMentions, dtype: float64


In [10]:
pd.set_option('display.max_colwidth', None)
df = pd.read_csv('project/01_data/raw/gdelt_raw.csv')
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'])
# 2024-05-23에 대만 관련이지만 좌표가 이상한 것들
mask = (df['SQLDATE'] == '2024-05-23') & (df['ActionGeo_Lat'] < 0)
print(df[mask][['NumMentions','ActionGeo_Lat','ActionGeo_Long','SOURCEURL']].to_string())


      NumMentions  ActionGeo_Lat  ActionGeo_Long                                                                                                                                        SOURCEURL
2606          298       -35.2833         149.217  https://www.hometownregister.com/news/national/chinas-military-surrounds-taiwan-as-punishment/article_9a75cd7e-585f-5603-bbd0-398801a1a26e.html


In [16]:
import pandas as pd
import numpy as np
from datetime import timedelta
from shapely.geometry import Point, LineString
import geopy.distance
import yaml

with open('config.yaml', encoding='utf-8') as f:
    config = yaml.safe_load(f)

geo_bins = config['priority_score']['geo_distance_bins']
MEDIAN_LINE = LineString([(122.0, 27.0), (118.0, 23.0)])

KNOWN_CRISES = [
    ('2025-12-29', '2025-12-30'),
    ('2025-04-01', '2025-04-02'),
    ('2024-10-14', '2024-10-14'),
    ('2024-05-23', '2024-05-24'),
    ('2024-05-20', '2024-05-20'),
    ('2024-01-13', '2024-01-13'),
    ('2023-08-12', '2023-08-18'),
    ('2023-04-05', '2023-04-10'),
    ('2022-08-02', '2022-08-10'),
    ('2021-10-01', '2021-10-04'),
    ('2021-03-26', '2021-03-26'),
    ('2020-09-17', '2020-09-19'),
    ('2020-08-09', '2020-08-12'),
    ('2019-03-31', '2019-03-31'),
    ('2018-04-18', '2018-04-18'),
    ('2016-05-20', '2016-05-20'),
    ('2013-11-23', '2013-11-23'),
]

df = pd.read_csv('project/01_data/raw/gdelt_raw.csv')
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'])
df = df.dropna(subset=['ActionGeo_Lat', 'ActionGeo_Long'])

def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

df['s_mentions'] = minmax(df['NumMentions'])
df['s_tone'] = np.where(df['AvgTone'] < -3, minmax(df['AvgTone'].abs()), 0)
df['priority_score'] = df['s_mentions'] * 0.8 + df['s_tone'] * 0.2

data_start = df['SQLDATE'].min().date()
data_end   = df['SQLDATE'].max().date()

crisis_dates = set()
for start, end in KNOWN_CRISES:
    s = pd.to_datetime(start).date()
    e = pd.to_datetime(end).date()
    for i in range((e-s).days+1):
        d = s + timedelta(days=i)
        if data_start <= d <= data_end:
            crisis_dates.add(d)

print('유효 위기 날짜:', len(crisis_dates), '일')
print()
header = 'Threshold | Precision |   Recall |       F1 |   TP |   FP |   FN'
print(header)
print('-' * 65)
for thr in np.arange(0.05, 0.95, 0.05).round(2):
    predicted = set(df[df['priority_score'] >= thr]['SQLDATE'].dt.date)
    tp = len(predicted & crisis_dates)
    fp = len(predicted - crisis_dates)
    fn = len(crisis_dates - predicted)
    p  = tp/(tp+fp) if (tp+fp) > 0 else 0
    r  = tp/(tp+fn) if (tp+fn) > 0 else 0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0
    print('{:>9.2f} | {:>9.3f} | {:>8.3f} | {:>8.3f} | {:>4} | {:>4} | {:>4}'.format(thr, p, r, f1, tp, fp, fn))

유효 위기 날짜: 47 일

Threshold | Precision |   Recall |       F1 |   TP |   FP |   FN
-----------------------------------------------------------------
     0.05 |     0.035 |    0.660 |    0.066 |   31 |  867 |   16
     0.10 |     0.140 |    0.468 |    0.216 |   22 |  135 |   25
     0.15 |     0.293 |    0.362 |    0.324 |   17 |   41 |   30
     0.20 |     0.343 |    0.255 |    0.293 |   12 |   23 |   35
     0.25 |     0.348 |    0.170 |    0.229 |    8 |   15 |   39
     0.30 |     0.333 |    0.149 |    0.206 |    7 |   14 |   40
     0.35 |     0.389 |    0.149 |    0.215 |    7 |   11 |   40
     0.40 |     0.375 |    0.128 |    0.190 |    6 |   10 |   41
     0.45 |     0.333 |    0.085 |    0.136 |    4 |    8 |   43
     0.50 |     0.333 |    0.085 |    0.136 |    4 |    8 |   43
     0.55 |     0.375 |    0.064 |    0.109 |    3 |    5 |   44
     0.60 |     0.429 |    0.064 |    0.111 |    3 |    4 |   44
     0.65 |     0.500 |    0.064 |    0.113 |    3 |    3 |   44
     0.7

In [1]:
import pandas as pd
from datetime import timedelta

df = pd.read_csv('project/01_data/processed/final_priority_geo.csv')
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'])

KNOWN_CRISES = [
    ('2025-12-29', '2025-12-30', '중국 대만 주변 포위형 군사활동'),
    ('2025-04-01', '2025-04-02', 'Strait Thunder-2025A'),
    ('2025-02-27', '2025-02-27', '대만 해안 실사격 훈련'),
    ('2024-10-14', '2024-10-14', 'Joint Sword-2024B'),
    ('2024-05-23', '2024-05-24', 'Joint Sword-2024A'),
    ('2024-01-13', '2024-01-13', '라이칭더 총통 당선'),
    ('2023-08-12', '2023-08-18', '라이칭더 미국 경유 방문'),
    ('2023-04-05', '2023-04-10', 'Shandong·Joint Sword'),
    ('2022-08-31', '2022-08-31', '금문도 드론 격추'),
    ('2022-08-04', '2022-08-10', '대만 포위 실사격 훈련'),
    ('2022-08-02', '2022-08-03', 'Pelosi 대만 방문'),
    ('2022-07-30', '2022-07-30', 'Pelosi 방문 직전 군사훈련'),
    ('2021-10-01', '2021-10-04', '역대 최대 ADIZ 진입'),
    ('2021-01-19', '2021-01-19', '대만 군사 방어 훈련'),
    ('2020-12-31', '2020-12-31', '미국 대만해협 통과'),
    ('2020-09-17', '2020-09-19', '미국 국무차관 방문'),
    ('2020-08-09', '2020-08-12', '미국 보건장관 방문'),
    ('2019-03-31', '2019-03-31', '대만해협 중간선 침범'),
    ('2018-04-18', '2018-04-18', '대만해협 실사격 훈련'),
    ('2016-05-20', '2016-05-20', '차이잉원 취임')
]

print('사건명 | 데이터 존재 여부 | 최고 Score | 최고 NumMentions')
print('-' * 70)
for start, end, name in KNOWN_CRISES:
    s = pd.to_datetime(start).date()
    e = pd.to_datetime(end).date()
    mask = (df['SQLDATE'].dt.date >= s) & (df['SQLDATE'].dt.date <= e)
    subset = df[mask]
    if len(subset) == 0:
        print(f'❌ {name} | 없음 | - | -')
    else:
        max_score = subset['priority_score'].max()
        max_mentions = subset['NumMentions'].max()
        print(f'✅ {name} | {len(subset)}건 | {max_score:.3f} | {max_mentions}')

사건명 | 데이터 존재 여부 | 최고 Score | 최고 NumMentions
----------------------------------------------------------------------
✅ 중국 대만 주변 포위형 군사활동 | 25건 | 0.156 | 208
✅ Strait Thunder-2025A | 21건 | 0.190 | 214
✅ 대만 해안 실사격 훈련 | 7건 | 0.610 | 1026
✅ Joint Sword-2024B | 14건 | 0.097 | 96
✅ Joint Sword-2024A | 33건 | 0.111 | 100
✅ 라이칭더 총통 당선 | 3건 | 0.007 | 22
✅ 라이칭더 미국 경유 방문 | 2건 | 0.079 | 154
✅ Shandong·Joint Sword | 109건 | 0.227 | 424
✅ 금문도 드론 격추 | 24건 | 0.417 | 682
✅ 대만 포위 실사격 훈련 | 193건 | 0.829 | 1420
✅ Pelosi 대만 방문 | 38건 | 0.399 | 642
✅ Pelosi 방문 직전 군사훈련 | 4건 | 0.798 | 1465
✅ 역대 최대 ADIZ 진입 | 39건 | 0.526 | 840
✅ 대만 군사 방어 훈련 | 2건 | 0.766 | 1406
✅ 미국 대만해협 통과 | 3건 | 0.690 | 1190
✅ 미국 국무차관 방문 | 27건 | 0.249 | 463
✅ 미국 보건장관 방문 | 3건 | 0.003 | 16
✅ 대만해협 중간선 침범 | 2건 | 0.002 | 14
✅ 대만해협 실사격 훈련 | 7건 | 0.165 | 184
✅ 차이잉원 취임 | 7건 | 0.800 | 1468


In [2]:
import json
with open('project/01_data/raw/tle_eo_sar.json') as f:
    tle = json.load(f)
print('위성 수:', len(tle))

위성 수: 652


In [3]:
import pandas as pd
df = pd.read_csv('project/01_data/processed/final_priority_geo.csv')
print('전체:', len(df))
print('중복 제거 후:', len(df.drop_duplicates(subset=['ActionGeo_Lat','ActionGeo_Long','SQLDATE'])))
print('Score 상위 500:', df.nlargest(500, 'priority_score')['SQLDATE'].nunique(), '일')


전체: 2548
중복 제거 후: 1394
Score 상위 500: 239 일


In [6]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
response = client.models.generate_content(model='gemini-2.5-flash', contents='안녕하세요')
print(response.text)

안녕하세요! 무엇을 도와드릴까요?


In [3]:
import pandas as pd

df=pd.read_csv(r'C:/Users/Jua/Desktop/DS/AIFFEL/00_AIFFELTHON/SIA_Project_Dash/project/01_data/processed/events_filtered.csv')

print('총 행:', len(df))
print('\n결측 비율(%):')
print((df[['Actor1Name','Actor2Name']].isna().mean()*100).round(1))
print('\nActor1Name 상위값:')
print(df['Actor1Name'].value_counts().head(10))
print('\nActor2Name 상위값:')
print(df['Actor2Name'].value_counts().head(10))

총 행: 2710

결측 비율(%):
Actor1Name    0.0
Actor2Name    0.0
dtype: float64

Actor1Name 상위값:
Actor1Name
CHINA         1124
TAIWAN         805
CHINESE        309
BEIJING        167
TAIPEI         166
TAIWANESE       82
XI JINPING      10
SHANGHAI         7
SHAANXI          7
TAICHUNG         5
Name: count, dtype: int64

Actor2Name 상위값:
Actor2Name
TAIWAN            1494
CHINA              612
CHINESE            343
BEIJING             93
TAIWANESE           71
TAIPEI              66
TAOYUAN              6
SHANGHAI             5
SHENZHEN             2
CHEN SHUI BIAN       2
Name: count, dtype: int64


In [1]:
import pandas as pd; df=pd.read_csv(r'C:/Users/Jua/Desktop/DS/AIFFEL/00_AIFFELTHON/SIA_Project_Dash/project/01_data/processed/events_filtered.csv'); bad=df[pd.to_numeric(df['GoldsteinScale'], errors='coerce').isna()]; print('문제 행 수:', len(bad)); print('컬럼 수:', len(df.columns)); print(bad[['GoldsteinScale','Actor1Name','Actor2Name']].head(10) if len(bad) else '문제 없음'); print('\nGoldsteinScale 샘플:'); print(df['GoldsteinScale'].head())

문제 행 수: 0
컬럼 수: 17
문제 없음

GoldsteinScale 샘플:
0    -9.5
1   -10.0
2   -10.0
3   -10.0
4   -10.0
Name: GoldsteinScale, dtype: float64


In [1]:
import sys; sys.path.insert(0,'project/03_dashboard'); from components.data_loader import df; print(df['reliability_grade'].value_counts())

reliability_grade
UNVERIFIED    890
HIGH          673
LOW           566
MEDIUM        427
Name: count, dtype: int64


In [2]:
import pandas as pd; df=pd.read_csv('project/01_data/processed/final_priority_geo.csv'); print('threshold 0.45 이상:', (df['priority_score']>=0.45).sum(), '/', len(df)); print(df['priority_score'].describe())

threshold 0.45 이상: 16 / 2556
count    2556.000000
mean        0.141875
std         0.091684
min         0.022500
25%         0.076341
50%         0.119728
75%         0.179524
max         0.641164
Name: priority_score, dtype: float64
